In [ ]:
# === Common notebook preamble (load llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for locating the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates when running from the notebooks folder
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === End preamble ===


# Ch 09. Optimization Algorithms — From SGD to Lion ⭐

> **Learning Objectives**
> - Derive the formulas for SGD, Momentum, RMSProp, Adam, AdamW, and Lion
> - Explain the meaning of the first moment $\mathbf{m}$ and second moment $\mathbf{v}$
> - Implement learning-rate schedulers (Warmup, Cosine, WSD)
> - Compare convergence speed and memory usage across optimizers

## 9.1 SGD and Mini-batches

**Stochastic Gradient Descent**:
$$\theta_{t+1} = \theta_t - \eta \, \mathbf{g}_t, \quad \mathbf{g}_t = \nabla_\theta \mathcal{L}(\theta_t; \mathcal{B}_t)$$

- $\mathcal{B}_t$: the $t$-th mini-batch
- $\eta$: learning rate

Estimate the gradient from a mini-batch instead of the full dataset → fast, but noisy.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Test functions
def bowl(x, y):
    return x**2 + 5 * y**2  # Elliptical bowl; the y direction is steeper

def bowl_grad(x, y):
    return np.array([2*x, 10*y])

# SGD
def sgd(grad_fn, x0, lr=0.1, n_steps=100):
    path = [np.array(x0, dtype=float)]
    x = np.array(x0, dtype=float)
    for _ in range(n_steps):
        g = grad_fn(*x)
        x = x - lr * g
        path.append(x.copy())
    return np.array(path)

path_sgd = sgd(bowl_grad, [2.0, 2.0], lr=0.1, n_steps=50)
print(f"SGD after 50 steps: {path_sgd[-1]} (minimum point (0, 0))")
print(f"Final loss: {bowl(*path_sgd[-1]):.2e}")


## 9.2 Momentum — A Physical Analogy of Inertia

$$\mathbf{v}_t = \beta \mathbf{v}_{t-1} + \mathbf{g}_t$$
$$\theta_{t+1} = \theta_t - \eta \, \mathbf{v}_t$$

- $\beta \approx 0.9$: exponentially weighted moving average of past gradients
- Reduces oscillation in steep directions and accelerates along gentle directions
- Physical analogy: a ball rolls with inertia


In [ ]:
# Momentum
def momentum(grad_fn, x0, lr=0.1, beta=0.9, n_steps=100):
    path = [np.array(x0, dtype=float)]
    x = np.array(x0, dtype=float)
    v = np.zeros_like(x)
    for _ in range(n_steps):
        g = grad_fn(*x)
        v = beta * v + g
        x = x - lr * v
        path.append(x.copy())
    return np.array(path)

path_momentum = momentum(bowl_grad, [2.0, 2.0], lr=0.1, beta=0.9, n_steps=50)
print(f"Momentum after 50 steps: {path_momentum[-1]}")
print(f"Final loss: {bowl(*path_momentum[-1]):.2e}")

# Visualization
from llm_math.viz import plot_contour_and_path
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_contour_and_path(axes[0], lambda v: bowl(*v), path_sgd, xlim=(-3, 3), ylim=(-3, 3),
                      title='SGD (strong oscillation)')
plot_contour_and_path(axes[1], lambda v: bowl(*v), path_momentum, xlim=(-3, 3), ylim=(-3, 3),
                      title='Momentum (faster convergence from inertia)')
plt.tight_layout()
plt.savefig('../figures/ch09_sgd_vs_momentum.png', dpi=100, bbox_inches='tight')
plt.show()


## 9.3 Adaptive Learning Rates — AdaGrad and RMSProp

**AdaGrad**: different learning rates per parameter. Frequently updated parameters get smaller steps.
$$\mathbf{r}_t = \mathbf{r}_{t-1} + \mathbf{g}_t^2$$
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\mathbf{r}_t + \epsilon}} \mathbf{g}_t$$

Problem: $\mathbf{r}$ keeps growing, so learning can stop.

**RMSProp**: fixes this with an exponential moving average
$$\mathbf{v}_t = \beta \mathbf{v}_{t-1} + (1-\beta) \mathbf{g}_t^2$$
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\mathbf{v}_t + \epsilon}} \mathbf{g}_t$$

## 9.4 Adam — First and Second Moments

**Adam** = Momentum + RMSProp
$$\mathbf{m}_t = \beta_1 \mathbf{m}_{t-1} + (1-\beta_1) \mathbf{g}_t$$  (first moment)
$$\mathbf{v}_t = \beta_2 \mathbf{v}_{t-1} + (1-\beta_2) \mathbf{g}_t^2$$  (second moment)

Bias correction (because the initial value is 0, early steps are biased small):
$$\hat{\mathbf{m}}_t = \frac{\mathbf{m}_t}{1 - \beta_1^t}, \quad \hat{\mathbf{v}}_t = \frac{\mathbf{v}_t}{1 - \beta_2^t}$$

Update:
$$\theta_{t+1} = \theta_t - \eta \frac{\hat{\mathbf{m}}_t}{\sqrt{\hat{\mathbf{v}}_t} + \epsilon}$$

- $\beta_1 = 0.9, \beta_2 = 0.999, \epsilon = 10^{-8}$ (defaults)
- The de facto standard for LLM training


In [ ]:
# Direct Adam implementation
class Adam:
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = params  # dict of arrays
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.t = 0

    def step(self, grads):
        self.t += 1
        for k in self.params:
            self.m[k] = self.beta1 * self.m[k] + (1 - self.beta1) * grads[k]
            self.v[k] = self.beta2 * self.v[k] + (1 - self.beta2) * grads[k]**2
            m_hat = self.m[k] / (1 - self.beta1**self.t)
            v_hat = self.v[k] / (1 - self.beta2**self.t)
            self.params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

# Test on a simple 2D function
def adam_optimize(grad_fn, x0, lr=0.1, n_steps=100):
    x = np.array(x0, dtype=float)
    params = {'x': x}
    opt = Adam(params, lr=lr)
    path = [x.copy()]
    for _ in range(n_steps):
        g = grad_fn(*x)
        opt.step({'x': g})
        x = params['x']
        path.append(x.copy())
    return np.array(path)

path_adam = adam_optimize(bowl_grad, [2.0, 2.0], lr=0.1, n_steps=50)
print(f"Adam after 50 steps: {path_adam[-1]}")
print(f"Final loss: {bowl(*path_adam[-1]):.2e}")

# Compare four optimizers
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (path, name) in zip(axes.flat, [
    (path_sgd, 'SGD'),
    (path_momentum, 'Momentum'),
    (path_adam, 'Adam'),
]):
    plot_contour_and_path(ax, lambda v: bowl(*v), path, xlim=(-3, 3), ylim=(-3, 3),
                          title=f'{name} (loss={bowl(*path[-1]):.2e})')
# Last empty axis
axes[1, 1].axis('off')
plt.tight_layout()
plt.savefig('../figures/ch09_optimizers_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


## 9.5 AdamW — Correctly Decoupling Weight Decay

If weight decay is simply added to Adam's gradient, it becomes entangled with the moments and does not have the intended effect.

**AdamW** (Decoupled Weight Decay):
$$\theta_{t+1} = \theta_t - \eta \left( \frac{\hat{\mathbf{m}}_t}{\sqrt{\hat{\mathbf{v}}_t} + \epsilon} + \lambda \theta_t \right)$$

- Do not put the weight decay term $\lambda \theta_t$ into the gradient; subtract it directly
- The real standard for LLM training (GPT and LLaMA both use AdamW)


In [ ]:
# AdamW vs Adam (L2 regularization)
class AdamW:
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.wd = weight_decay
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.t = 0

    def step(self, grads):
        self.t += 1
        for k in self.params:
            self.m[k] = self.beta1 * self.m[k] + (1 - self.beta1) * grads[k]
            self.v[k] = self.beta2 * self.v[k] + (1 - self.beta2) * grads[k]**2
            m_hat = self.m[k] / (1 - self.beta1**self.t)
            v_hat = self.v[k] / (1 - self.beta2**self.t)
            # Apply weight decay directly instead of adding it to the gradient
            self.params[k] -= self.lr * (m_hat / (np.sqrt(v_hat) + self.eps) + self.wd * self.params[k])

# AdamW effect: weight magnitude decreases
print("AdamW weight-decay effect:")
params = {'w': np.array([10.0, 10.0, 10.0])}
opt = AdamW(params, lr=0.1, weight_decay=0.1)
for i in range(10):
    grad = {'w': np.array([0.01, 0.01, 0.01])}  # Small gradient
    opt.step(grad)
    print(f"  step {i+1}: w = {params['w'].round(4)}")
print("\n=> Weights decrease toward 0 (weight decay).")


## 9.6 Learning-Rate Schedulers

Dynamic schedules are usually better than a fixed learning rate.

- **Warmup**: increase linearly for the first $N$ steps. This gives a stable start.
- **Cosine**: $\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})(1 + \cos(\pi t / T))$
- **WSD (Warmup-Stable-Decay)**: used in recent LLM training. Keep a stable phase, then decay rapidly.


In [ ]:
# Implement and visualize schedulers
def constant_lr(t, max_t, lr_max=1.0, warmup=0):
    return lr_max

def cosine_lr(t, max_t, lr_max=1.0, lr_min=0.0, warmup=0):
    if t < warmup:
        return lr_max * t / warmup
    progress = (t - warmup) / (max_t - warmup)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * progress))

def wsd_lr(t, max_t, lr_max=1.0, lr_min=0.0, warmup=100, stable_end=0.8):
    """Warmup-Stable-Decay."""
    if t < warmup:
        return lr_max * t / warmup
    elif t < stable_end * max_t:
        return lr_max
    else:
        progress = (t - stable_end * max_t) / (max_t - stable_end * max_t)
        return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * progress))

t = np.arange(1000)
fig, ax = plt.subplots(figsize=(11, 5))
for name, fn in [
    ('Constant', lambda t: constant_lr(t, 1000, 1.0, 100)),
    ('Cosine', lambda t: cosine_lr(t, 1000, 1.0, 0.0, 100)),
    ('WSD', lambda t: wsd_lr(t, 1000, 1.0, 0.0, 100, 0.7)),
]:
    ax.plot(t, [fn(ti) for ti in t], label=name, linewidth=2)
ax.set_xlabel('Step'); ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Scheduler Comparison')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch09_lr_schedules.png', dpi=100, bbox_inches='tight')
plt.show()


## 9.7 Gradient Clipping

$$\mathbf{g} \leftarrow \frac{\mathbf{g}}{\max(1, \|\mathbf{g}\| / c)}$$

Prevents exploding gradients. $c$ is usually 1.0. Essential in LLM training.


In [ ]:
# Gradient clipping
def clip_grad_norm(grad, max_norm=1.0):
    norm = np.linalg.norm(grad)
    if norm > max_norm:
        grad = grad * (max_norm / norm)
    return grad, norm

# Test
np.random.seed(0)
small_grad = np.random.randn(3) * 0.1
big_grad = np.random.randn(3) * 100

clipped_small, norm_small = clip_grad_norm(small_grad, 1.0)
clipped_big, norm_big = clip_grad_norm(big_grad, 1.0)

print(f"Small gradient: norm={norm_small:.4f} -> norm after clipping={np.linalg.norm(clipped_small):.4f} (unchanged)")
print(f"Large gradient:   norm={norm_big:.4f} -> norm after clipping={np.linalg.norm(clipped_big):.4f} (limited to 1.0)")


## 9.8 [CPU/GPU Benchmark ③] Optimizer Convergence and Time Comparison

Let's compare optimizers in PyTorch.


In [ ]:
# Optimizer comparison (PyTorch)
import torch
import torch.nn as nn
from llm_math.data import load_mnist_small

X_train, y_train, X_test, y_test = load_mnist_small(n_train=3000, n_test=1000)
X_t = torch.tensor(X_train, dtype=torch.float32)
y_t = torch.tensor(y_train, dtype=torch.long)

def train_with_optim(opt_name, n_epochs=5, batch_size=64):
    torch.manual_seed(0)
    model = nn.Sequential(
        nn.Linear(784, 128), nn.ReLU(),
        nn.Linear(128, 64), nn.ReLU(),
        nn.Linear(64, 10)
    )
    if opt_name == 'SGD':
        opt = torch.optim.SGD(model.parameters(), lr=0.1)
    elif opt_name == 'Momentum':
        opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
    elif opt_name == 'Adam':
        opt = torch.optim.Adam(model.parameters(), lr=0.001)
    elif opt_name == 'AdamW':
        opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

    loss_fn = nn.CrossEntropyLoss()
    losses = []
    for epoch in range(n_epochs):
        idx = torch.randperm(len(X_t))
        for i in range(0, len(X_t), batch_size):
            bi = idx[i:i+batch_size]
            xb, yb = X_t[bi], y_t[bi]
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            opt.step()
            losses.append(loss.item())
    return losses

# Run all optimizers
results = {}
for name in ['SGD', 'Momentum', 'Adam', 'AdamW']:
    print(f"Training with {name}...")
    losses = train_with_optim(name, n_epochs=3)
    results[name] = losses
    print(f"  final loss: {losses[-1]:.4f}")

# Visualization
fig, ax = plt.subplots(figsize=(11, 5))
for name, losses in results.items():
    # Smooth with a moving average
    window = 50
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, label=name, linewidth=2)
ax.set_xlabel('Step (with smoothing applied)')
ax.set_ylabel('Loss')
ax.set_title('Convergence Speed by Optimizer (MNIST MLP)')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('../figures/ch09_optimizers_mnist.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n=> Adam/AdamW converge much faster than SGD. AdamW is the standard for LLM training.")


## 9.9 Key Takeaways

| Optimizer | Core idea | Memory per parameter |
|---|---|---|
| SGD | Plain gradient | 0 |
| Momentum | First moment (inertia) | 1x |
| RMSProp | Second moment (adaptive lr) | 1x |
| Adam | First + second moments | 2x |
| AdamW | Adam + decoupled weight decay | 2x |

## Exercises

1. Apply SGD, Momentum, and Adam to the Rosenbrock function and compare convergence paths.
2. Show why Adam's bias correction is needed by comparing the magnitude of $\mathbf{m}_t$ in early steps.
3. Compare Adam vs AdamW on MNIST with weight decay set to 0.01.
4. Run Adam with learning rates $10^{-2}, 10^{-3}, 10^{-4}$ and compare convergence speed.
5. Explain why LLM training becomes unstable without a warmup phase.

> Solutions: `solutions/ch09_solutions.ipynb`
